# Solafune Satellite Precipitation Nowcasting — Kaggle Runner

Trains the Solafune nowcasting solution end-to-end on a Kaggle GPU (T4/P100/L4).

**Datasets to attach**
1. `solafune-precip` — the competition data (train + eval + placeholders)
2. `solafune-repo`   — this repository as a Kaggle Dataset (upload the whole `D:/solafune` tree)

**Run order**
1. Setup — installs, path wiring
2. Cache benchmark — picks Zarr vs memmap
3. Build train + eval caches (once per session; resumes cleanly)
4. Compute normalization stats
5. Train (change `EXPERIMENT` to switch configs)
6. Inference + submission

**Session-safe** — if the 12h wall hits mid-training, re-run cell 5; `try_auto_resume()` picks up `last.pt`.

## 1. Setup

In [ ]:
# Repo is expected at /kaggle/input/solafune-repo (a Kaggle Dataset containing the D:/solafune tree)
REPO_DIR = '/kaggle/input/solafune-repo'
DATA_ROOT = '/kaggle/input/solafune-precip'
OUT_DIR = '/kaggle/working'

import os, sys, subprocess
sys.path.insert(0, REPO_DIR)

# The competition allows Solafune-Tools + standard PyTorch stack. Install anything not on Kaggle by default.
for pkg in ('zarr', 'numcodecs', 'hydra-core', 'omegaconf'):
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Auto-detect the actual data root — Solafune ships nested directories
from pathlib import Path
def _find(root, name_startswith):
    root = Path(root)
    if any(p.name.startswith(name_startswith) for p in root.iterdir()):
        return next(p for p in root.iterdir() if p.name.startswith(name_startswith))
    for p in root.rglob('*'):
        if p.is_dir() and p.name.startswith(name_startswith):
            return p
    raise FileNotFoundError(f'no dir starting with {name_startswith!r} under {root}')

TRAIN_ROOT = _find(DATA_ROOT, 'train_dataset')
EVAL_ROOT = _find(DATA_ROOT, 'evaluation_dataset')
TRAIN_CSV = TRAIN_ROOT / 'train_dataset.csv'
EVAL_CSV  = EVAL_ROOT / 'evaluation_target.csv'
CACHE_ROOT = Path(OUT_DIR) / 'cache'
NORM_STATS = CACHE_ROOT / 'norm_stats.json'

print('TRAIN_ROOT:', TRAIN_ROOT)
print('EVAL_ROOT: ', EVAL_ROOT)
assert TRAIN_CSV.exists() and EVAL_CSV.exists(), 'CSVs missing — verify DATA_ROOT'

## 2. Cache Backend Benchmark

In [ ]:
from src.data.cache.benchmark import run_benchmark
backend_json = CACHE_ROOT / 'backend.json'
if not backend_json.exists():
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    result = run_benchmark(output_path=backend_json, n_samples=200, channels=10, hw=81)
    print('recommended backend:', result.recommended)

import json
BACKEND = json.loads(backend_json.read_text())['recommended']
print('using backend:', BACKEND)

# On Kaggle we prefer Zarr for compression (disk-limited); switch here to override.
BACKEND = 'zarr'
print('locked backend to:', BACKEND)

## 3. Build Train + Eval Caches

In [ ]:
import pandas as pd
from src.data.cache import get_backend
from src.data.preprocessing import build_cache, build_cache_spec

def _build_cache_if_missing(csv, root, out_dir, load_gpm):
    spec_path = out_dir / 'spec.json'
    if spec_path.exists():
        print(f'cache exists: {out_dir}')
        return
    df = pd.read_csv(csv)
    spec, _ = build_cache_spec(df, out_dir, 'ir_only')
    cls = get_backend(BACKEND)
    b = cls(spec, compressor='lz4') if BACKEND == 'zarr' else cls(spec)
    build_cache(csv, root, out_dir, b, 'ir_only', load_gpm=load_gpm, verbose_every=2000)
    b.close()
    print(f'cache built: {out_dir}')

_build_cache_if_missing(TRAIN_CSV, TRAIN_ROOT, CACHE_ROOT / 'train', load_gpm=True)
_build_cache_if_missing(EVAL_CSV,  EVAL_ROOT,  CACHE_ROOT / 'eval',  load_gpm=False)

## 4. Normalization Statistics

In [ ]:
from src.constants import SATELLITES
from src.data.normalization import compute_norm_stats, save_norm_stats
from src.paths import sat_tif_path
from src.utils import parse_frame_list

if not NORM_STATS.exists():
    train_df = pd.read_csv(TRAIN_CSV)
    paths = {s: [] for s in SATELLITES}
    for _, row in train_df.iterrows():
        s = row['satellite_target']
        for f in parse_frame_list(row['last_30_minutes_observation_filename']):
            paths[s].append(sat_tif_path(TRAIN_ROOT, s, f))
    for s in SATELLITES:
        print(s, 'candidate TIFs:', len(paths[s]))
    stats = compute_norm_stats(paths, max_files_per_satellite=500, pixel_stride=2)
    save_norm_stats(NORM_STATS, stats)
print('norm stats:', NORM_STATS)

## 5. Training

Change `EXPERIMENT` and rerun this cell to iterate through the frozen experimental roadmap.

| EXPERIMENT | Hydra overrides applied |
|---|---|
| `exp0_baseline` | strong baseline (log1p + rain-weighted + BCE + stratified sampling + EMA) |
| `exp1_rain_weighted` | `loss.rain_weighted=true loss.rain_weight_scale=3.0` |
| `exp2_conv3d` | `+model/temporal=conv3d` |
| `exp3_nll` | `+model/heads=probabilistic loss.nll_weight=0.3` |
| `exp4_sampling` | `data.sampling.strategy=combined` (loc × precip) |
| `exp5_band_dropout` | `augmentation=full` with `band_dropout_prob=0.3` |
| `exp6_efficientnet` | `+model/encoder=efficientnet_b3` |
| `exp7_seed_ensemble` | second seed (run with `SEED=43`) — then average at inference |

In [ ]:
EXPERIMENT = 'exp0_baseline'    # <── change here to sweep experiments
EPOCHS     = 50
SEED       = 42

TRAINING_DIR = Path(OUT_DIR) / 'training' / EXPERIMENT / f'seed_{SEED}'
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

# Compose config via Hydra
from hydra import compose, initialize_config_dir
import hydra
hydra_dir = str(Path(REPO_DIR) / 'configs')
if hydra.core.global_hydra.GlobalHydra.instance().is_initialized():
    hydra.core.global_hydra.GlobalHydra.instance().clear()

overrides = [
    f'data.cache_dir={CACHE_ROOT}/train',
    f'data.norm_stats_path={NORM_STATS}',
    f'data.train_csv={TRAIN_CSV}',
    f'data.eval_csv={EVAL_CSV}',
    f'data.train_root={TRAIN_ROOT}',
    f'data.eval_root={EVAL_ROOT}',
    f'data.cache.backend={BACKEND}',
    f'training.output_dir={TRAINING_DIR}',
    f'training.epochs={EPOCHS}',
    f'seed={SEED}',
    f'+experiment={EXPERIMENT}',
]
with initialize_config_dir(config_dir=hydra_dir, version_base=None):
    cfg = compose(config_name='config', overrides=overrides)

from omegaconf import OmegaConf
print(OmegaConf.to_yaml(cfg))

In [ ]:
# Build datasets, model, trainer
from src.constants import max_active_channels
from src.data.dataloader import DataLoaderConfig, build_dataloader, build_sampler
from src.data.dataset import DatasetConfig, SolafuneDataset, split_indices_by_location
from src.experiment.tracker import snapshot_run
from src.models import build_model
from src.seed import seed_everything
from src.training.losses import build_loss
from src.training.schedulers import build_optimizer, build_scheduler
from src.training.trainer import Trainer, TrainerConfig

seed_everything(int(cfg.seed))
snapshot_run(TRAINING_DIR, cfg, repo_root=Path(REPO_DIR))

ds_cfg = DatasetConfig(
    cache_dir=Path(cfg.data.cache_dir),
    csv_path=Path(cfg.data.train_csv),
    norm_stats_path=Path(cfg.data.norm_stats_path),
    image_size=int(cfg.data.image_size),
    bands=str(cfg.data.bands),
    include_diff_frames=bool(cfg.data.include_diff_frames),
    missing_frame_strategy=str(cfg.data.missing_frame_strategy),
    cache_backend=BACKEND,
)
df = pd.read_csv(ds_cfg.csv_path)
train_idx, val_idx = split_indices_by_location(df, list(cfg.data.val_locations))
train_ds = SolafuneDataset(ds_cfg, df=df, indices=train_idx)
val_ds   = SolafuneDataset(ds_cfg, df=df, indices=val_idx)
print(f'train: {len(train_ds)}   val: {len(val_ds)}')

dl = DataLoaderConfig(
    batch_size=int(cfg.data.batch_size), num_workers=int(cfg.data.num_workers),
    pin_memory=True, persistent_workers=True, prefetch_factor=2, drop_last=True,
)
train_loader = build_dataloader(
    train_ds, dl,
    sampler=build_sampler(train_ds, str(cfg.data.sampling.strategy),
                          precip_weight_scale=float(cfg.data.sampling.precip_weight_scale)),
    base_seed=int(cfg.seed),
)
val_loader = build_dataloader(val_ds, dl, shuffle=False, base_seed=int(cfg.seed))

mcfg = OmegaConf.to_container(cfg.model, resolve=True)
mcfg['in_channels_per_frame'] = max_active_channels(str(cfg.data.bands))
mcfg['n_frames'] = int(cfg.data.frames)
mcfg['n_diff_frames'] = int(cfg.data.frames - 1) if cfg.data.include_diff_frames else 0
model = build_model(mcfg)
print(f'params: {sum(p.numel() for p in model.parameters()):,}')

loss_fn = build_loss(OmegaConf.to_container(cfg.loss))
opt = build_optimizer(model, OmegaConf.to_container(cfg.optimizer))
sched, seb = build_scheduler(
    opt, OmegaConf.to_container(cfg.scheduler),
    steps_per_epoch=len(train_loader), epochs=int(cfg.training.epochs),
)
tcfg = TrainerConfig(**OmegaConf.to_container(cfg.training))
tcfg.step_scheduler_each_batch = seb
trainer = Trainer(model, opt, sched, loss_fn, train_loader, val_loader, tcfg)
trainer.try_auto_resume()   # session-safe: pick up last.pt if present

In [ ]:
# Kick off training. Progress is streamed via the logger + saved to CSV/TensorBoard.
best_val = trainer.fit()
print('best val metric:', best_val)

## 6. Diagnostic plots

In [ ]:
from src.visualization import plot_training_curves, plot_val_curves
plot_training_curves(TRAINING_DIR / 'train_metrics.csv', TRAINING_DIR / 'plots' / 'train.png')
plot_val_curves(TRAINING_DIR / 'val_metrics.csv', TRAINING_DIR / 'plots' / 'val.png')
print('plots saved in', TRAINING_DIR / 'plots')

## 7. Inference + Submission

In [ ]:
from src.inference.predict import PredictionConfig, predict
from src.inference.submission import write_submission
from src.training.callbacks import CheckpointSaver

# Load best (EMA-preferred) checkpoint
best_ckpt = TRAINING_DIR / 'checkpoints' / 'best.pt'
assert best_ckpt.exists(), best_ckpt
state = torch.load(best_ckpt, map_location='cpu', weights_only=False)
if state.get('ema') is not None:
    # Load EMA weights (shadow → model)
    from src.training.ema import ExponentialMovingAverage
    ema = ExponentialMovingAverage(model, decay=cfg.training.ema_decay)
    ema.load_state_dict(state['ema'])
    ema.apply(model).__enter__()   # leave model in EMA weights for inference
    print('loaded EMA weights')
else:
    model.load_state_dict(state['model'])
    print('loaded raw weights')

# Build eval dataset
eval_ds_cfg = DatasetConfig(
    cache_dir=CACHE_ROOT / 'eval',
    csv_path=EVAL_CSV,
    norm_stats_path=NORM_STATS,
    image_size=int(cfg.data.image_size), bands='ir_only',
    include_diff_frames=True, cache_backend=BACKEND,
)
eval_ds = SolafuneDataset(eval_ds_cfg)
eval_loader = build_dataloader(
    eval_ds,
    DataLoaderConfig(batch_size=32, num_workers=2, pin_memory=True,
                     persistent_workers=True, prefetch_factor=2, drop_last=False),
    shuffle=False, base_seed=int(cfg.seed),
)
print(f'eval samples: {len(eval_ds)}')

preds = predict(
    model, eval_loader,
    PredictionConfig(amp=True, tta=True, rain_mask_threshold=0.15),
)
print(f'predictions shape: {preds.shape}   finite: {bool((preds == preds).all())}   min/max: {preds.min():.3f}/{preds.max():.3f}')

In [ ]:
# Write submission GeoTIFFs (identity CRS, float32, 41×41, LZW-compressed)
SUB_DIR = Path(OUT_DIR) / 'submission'
TEST_FILES = SUB_DIR / 'test_files'
n_written = write_submission(preds, EVAL_CSV, TEST_FILES)

# Copy evaluation_target.csv into the submission directory per Solafune format
import shutil
shutil.copy(EVAL_CSV, SUB_DIR / 'evaluation_target.csv')

# Zip archive for upload
archive = shutil.make_archive(str(Path(OUT_DIR) / 'submission'), 'zip', str(SUB_DIR))
print(f'files written: {n_written}')
print(f'archive: {archive}   size: {Path(archive).stat().st_size / 1e6:.1f} MB')

In [ ]:
# Final format audit — reads a random subset back and asserts the Solafune contract
import random, numpy as np
from src.utils.io import read_gpm_tif
random.seed(0)
sample_names = random.sample(list(pd.read_csv(EVAL_CSV)['gpm_imerg_filename']), 25)
for name in sample_names:
    arr, meta = read_gpm_tif(TEST_FILES / name)
    assert arr.shape == (41, 41), (name, arr.shape)
    assert arr.dtype == np.float32, (name, arr.dtype)
    assert np.isfinite(arr).all(), name
print(f'audit passed: 25/25 sampled TIFs conform to (41,41) float32 finite')
print('submission ready at:', SUB_DIR)

## 8. Experiment sweep helper (optional)

Runs the frozen roadmap sequentially inside a single Kaggle session. Each experiment
starts from the previous best checkpoint via `try_auto_resume()`.

**Warning**: A full sweep is ~40+ GPU hours. Only enable if you have a fresh session and expect the compute budget.

```python
SWEEP = ['exp0_baseline', 'exp2_conv3d', 'exp3_nll']
for exp in SWEEP:
    print(f'\n=== running {exp} ===')
    # 1. rerun cell 5 with EXPERIMENT=exp
    # 2. rerun cell 7 to produce a per-experiment submission
    # (in a notebook: just set EXPERIMENT and click Run All from cell 5 downward)
```